# 01 — Exploratory Data Analysis

This notebook explores the acoustic event dataset used to train the classifier.

**Contents:**
1. **Class distribution** — sample counts per class.
2. **Waveforms** — one example time-series per class.
3. **Log-Mel spectrograms** — one 2-D frequency-time representation per class (shared colour scale).

> **Pre-requisite:** Run `src/preprocess.py` and `src/features.py` to populate
> `data/processed/` and `data/spectrograms/` before executing this notebook.

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so src.* imports resolve from the notebooks/ directory.
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

from src.config import CLASS_LABELS, HOP_LENGTH, SAMPLE_RATE
from src.features import compute_melspectrogram

DATA_DIR  = ROOT / "data"
SPEC_DIR  = DATA_DIR / "spectrograms"
PROC_DIR  = DATA_DIR / "processed"

print(f"ROOT     : {ROOT}")
print(f"SPEC_DIR : {SPEC_DIR}  (exists={SPEC_DIR.exists()})")
print(f"PROC_DIR : {PROC_DIR}  (exists={PROC_DIR.exists()})")

In [ ]:
labels_csv = SPEC_DIR / "labels.csv"

if not labels_csv.exists():
    print(f"WARNING: {labels_csv} not found.")
    print("Run:  python src/features.py --data-dir data/")
    df_orig = None
else:
    df = pd.read_csv(labels_csv)
    df["is_augmented"] = df["is_augmented"].astype(bool)
    df_orig = df[~df["is_augmented"]].copy()
    print(f"Non-augmented samples: {len(df_orig)}")
    print(df_orig["class_label"].value_counts().reindex(CLASS_LABELS.values()).to_string())

## Section 1 — Class Distribution

The dataset targets five acoustic event classes. A balanced distribution (similar
sample counts per class) reduces the risk of the model defaulting to a majority-class
prediction strategy. Class imbalance, if present, is compensated during training
via per-class weights passed to `model.fit`.

In [ ]:
if df_orig is None:
    print("No data — skipping class distribution plot.")
else:
    counts = df_orig["class_label"].value_counts().reindex(CLASS_LABELS.values())

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(counts.index, counts.values, color="steelblue", edgecolor="white", width=0.6)
    ax.bar_label(bars, padding=4, fontsize=11)
    ax.set_xlabel("Acoustic class")
    ax.set_ylabel("Sample count")
    ax.set_title("Class Distribution — non-augmented samples")
    ax.set_ylim(0, counts.max() * 1.15)
    ax.tick_params(axis="x", rotation=18)
    plt.tight_layout()
    plt.show()

## Section 2 — Waveform Analysis

Each plot shows the amplitude envelope of one 4-second preprocessed clip.

What to look for:
- **`normal_operation`** — low-amplitude, steady hum; flat and continuous.
- **`metallic_impact`** — sharp high-amplitude spike at the moment of impact, then rapid decay.
- **`friction_squeal`** — sustained oscillation with a narrower amplitude band than impact.
- **`alarm_tone`** — periodic on/off pattern clearly visible as repeating bursts.
- **`silence_ambient`** — near-zero amplitude throughout; occasional noise floor spikes.

In [ ]:
if df_orig is None:
    print("No data — skipping waveform plots.")
else:
    fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)

    for ax, (class_id, class_label) in zip(axes, CLASS_LABELS.items()):
        wav_files = (
            sorted((PROC_DIR / class_label).glob("*.wav"))
            if (PROC_DIR / class_label).exists()
            else []
        )
        if not wav_files:
            ax.set_title(f"{class_label}  (no files — run preprocess.py first)")
            continue
        audio, sr = sf.read(wav_files[0], dtype="float32")
        times = np.linspace(0, len(audio) / sr, num=len(audio))
        ax.plot(times, audio, linewidth=0.4, color="steelblue")
        ax.set_title(f"Class {class_id}: {class_label}", fontsize=10)
        ax.set_ylabel("Amplitude")
        ax.set_ylim(-1.1, 1.1)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle("Example Waveforms — one clip per class", fontsize=13)
    plt.tight_layout()
    plt.show()

## Section 3 — Log-Mel Spectrogram Analysis

Log-Mel spectrograms are the **primary input representation** fed to the CNN.
The x-axis is time (frames), the y-axis is frequency (Mel scale), and colour encodes
log energy. **All five plots share the same colour scale** so energy levels are directly
comparable across classes.

Key visual signatures to look for:
- `normal_operation` — broad, stable low-frequency energy.
- `metallic_impact` — brief broadband flash at the moment of impact.
- `friction_squeal` — concentrated bright horizontal stripe in the high-frequency region.
- `alarm_tone` — repeating vertical stripes reflecting the periodic on/off pattern.
- `silence_ambient` — uniformly dark, very low energy throughout the clip.

In [ ]:
# Collect one spectrogram per class; establish shared colour range (5th–95th percentile).
specs = {}
if df_orig is not None:
    for class_id, class_label in CLASS_LABELS.items():
        rows = df_orig[df_orig["class_label"] == class_label]
        if rows.empty:
            continue
        npy_path = Path(rows.iloc[0]["path"])
        if npy_path.exists():
            specs[class_label] = np.load(npy_path)

if not specs:
    print("No spectrograms found — run src/features.py first.")
else:
    all_values = np.concatenate([s.ravel() for s in specs.values()])
    vmin = float(np.percentile(all_values, 5))
    vmax = float(np.percentile(all_values, 95))

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    for ax, (class_id, class_label) in zip(axes, CLASS_LABELS.items()):
        if class_label not in specs:
            ax.set_title(f"{class_label}\n(missing)")
            ax.axis("off")
            continue
        img = librosa.display.specshow(
            specs[class_label],
            sr=SAMPLE_RATE,
            hop_length=HOP_LENGTH,
            x_axis="time",
            y_axis="mel",
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            cmap="magma",
        )
        ax.set_title(f"Class {class_id}\n{class_label}", fontsize=9)

    fig.colorbar(img, ax=axes.tolist(), format="%+.1f", label="Log energy")
    fig.suptitle(
        "Log-Mel Spectrograms — one clip per class (shared colour scale)", y=1.04
    )
    plt.tight_layout()
    plt.show()

## Observations

| Class | Visual signature | Likely challenge |
|---|---|---|
| `normal_operation` | Stable broadband low-frequency energy | May overlap with low-energy ambient |
| `metallic_impact` | Brief broadband transient flash | Short event may be missed in 4 s window |
| `friction_squeal` | Persistent bright high-frequency band | Could be confused with alarm tones |
| `alarm_tone` | Repeating vertical stripes (periodic on/off) | Frequency varies across alarm devices |
| `silence_ambient` | Uniformly dark, minimal energy | Easy to classify; serves as anchor class |

These visual differences motivate the CNN's 2-D convolutional filters, which detect
both frequency-band patterns and temporal structure simultaneously.